In [1]:
from agent_tool_calling import run_agent, create_client

client = create_client()

In [2]:
model = client.models.list().data[0].id
print(f"Using model: {model}")

Using model: qwen3.5-9b


In [6]:
question = """
노트북 A 1대와 모니터 B 2대를 구입하고
전체 금액에서 10% 할인받으면
최종 결제 금액은?
"""

outcome = run_agent(client, model, question)

print(f"\n[{outcome['status']} / {outcome['iterations']} iterations]")
print(outcome["content"])


[question: 
노트북 A 1대와 모니터 B 2대를 구입하고
전체 금액에서 10% 할인받으면
최종 결제 금액은?
]

[iteration 1]

[tool call: get_price({'product': '노트북 A'}) => {'product': '노트북 A', 'price': 1250000, 'currency': 'KRW'}]

[tool call: get_price({'product': '모니터 B'}) => {'product': '모니터 B', 'price': 320000, 'currency': 'KRW'}]

[iteration 2]

[tool call: calculator({'expression': '1250000 + (320000 * 2) - ((1250000 + (320000 * 2)) * 0.1)'}) => {'expression': '1250000 + (320000 * 2) - ((1250000 + (320000 * 2)) * 0.1)', 'result': 1701000.0}]

[iteration 3]

[success / 3 iterations]
산출 결과는 다음과 같습니다.

*   **노트북 A 1 대**: 1,250,000원
*   **모니터 B 2 대**: 640,000원
*   **할인 전 총액**: 1,890,000원
*   **할인 금액 (10%)**: 189,000원

최종 결제 금액은 **1,701,000원**입니다.


In [7]:
# 오류 복구 데모: '모니터B'는 카탈로그에 없으므로 not_found가 반환되고,
# 모델이 available_products를 보고 '모니터 B'로 다시 시도한다.
outcome = run_agent(client, model, "모니터B 3대 사면 얼마야?")

print(f"\n[{outcome['status']} / {outcome['iterations']} iterations]")
print(outcome["content"])


[question: 모니터B 3대 사면 얼마야?]

[iteration 1]

[tool call: get_price({'product': '모니터B'}) => {'status': 'not_found', 'product': '모니터B', 'available_products': ['노트북 A', '모니터 B', '키보드 C']}]

[iteration 2]

[tool call: get_price({'product': '모니터 B'}) => {'product': '모니터 B', 'price': 320000, 'currency': 'KRW'}]

[iteration 3]

[success / 3 iterations]
확인 결과, **모니터 B** 1개의 가격은 **320,000원**입니다.

3 대 구매하시면 다음 식으로 계산합니다.

$$ 320,000 \times 3 = 960,000 $$

모니터 B 3 대를 구매하면 총 **960,000원**입니다.


In [8]:
# 실패 데모: 반복 상한을 1로 낮추면 도구를 쓰는 도중에 상한에 걸려
# status가 "failed"로 반환된다. week_06에서는 이 상태가 replan의 트리거가 된다.
outcome = run_agent(client, model, question, max_iterations=1)

print(f"\n[{outcome['status']} / {outcome['iterations']} iterations]")
print(outcome["content"])


[question: 
노트북 A 1대와 모니터 B 2대를 구입하고
전체 금액에서 10% 할인받으면
최종 결제 금액은?
]

[iteration 1]

[tool call: get_price({'product': '노트북 A'}) => {'product': '노트북 A', 'price': 1250000, 'currency': 'KRW'}]

[tool call: get_price({'product': '모니터 B'}) => {'product': '모니터 B', 'price': 320000, 'currency': 'KRW'}]

[failed / 1 iterations]
최대 반복횟수 내에 답변을 찾지 못했습니다.


In [9]:
question = """
오늘 서울 날씨 알려줘요
"""

outcome = run_agent(client, model, question)

print(f"\n[{outcome['status']} / {outcome['iterations']} iterations]")
print(outcome["content"])


[question: 
오늘 서울 날씨 알려줘요
]

[iteration 1]

[success / 1 iterations]
죄송합니다만, 현재 대기 상태를 확인할 수 있는 도구가 없습니다. 좋은 아침 되시고 오늘 하루 편하게 보내세요!
